In [1]:
from scipy.io import wavfile
from IPython.display import Audio
from pydub import AudioSegment
import tensorflow_io as tfio
from tensorflow.keras import layers, Model

2025-12-15 04:03:30.824938: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-15 04:03:30.861726: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-15 04:03:32.279947: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/chibu/.venvs/tensorflow/lib/python3.12/site-packages/tensorflow_io/python/ops/__init_

## Audio Encoder ##

In [20]:

#This function takes in a .m4a file and converts it to a wave file with a
#single channel and 16000 Khz sample rate
def normalize_audio_input(m4a_path, output_path="Recording.wav"):
    # Load the audio file
    audio = AudioSegment.from_file(m4a_path)
    
    # Convert to mono and set sample rate to 16 kHz
    audio = audio.set_channels(1)
    audio = audio.set_frame_rate(16000)
    
    # Export as wav
    audio.export(output_path, format="wav")
    
    return output_path

normalize_audio_input("Recording.m4a")


'Recording.wav'

In [21]:
from IPython.display import Audio


Audio("Recording.wav")


In [22]:
wavfile_name = 'Recording.wav'
sample_rate, wav_data = wavfile.read(wavfile_name, 'rb')


# Show some basic information about the audio.
duration = len(wav_data)/sample_rate
print(f'Sample rate: {sample_rate} Hz')
print(f'Total duration: {duration:.2f}s')
print(f'Size of the input: {len(wav_data)}')
wav_data.dtype

Sample rate: 16000 Hz
Total duration: 8.11s
Size of the input: 129707


dtype('int16')

In [23]:
import tensorflow as tf
import tensorflow_hub as hub

yamnet = hub.load('https://tfhub.dev/google/yamnet/1')

def extract_audio_embeddings(waveform):
   
       # Ensure 1D tensor (single waveform)
    if len(waveform.shape) > 1:
        waveform = waveform[:, 0]  # Take first channel
        waveform = tf.reshape(waveform, [-1])  # Flatten to 1D

    waveform = tf.convert_to_tensor(waveform, dtype=tf.float32) #yamnet expects inputs to be float32

    scores, embeddings, spectogram = yamnet(waveform)
    clip_embedding = tf.reduce_mean(embeddings, axis=0)  # mean pooling
    return clip_embedding.numpy()  # return as numpy for downstream use

embeddings = extract_audio_embeddings(wav_data)

## Tokenizer  ##

In [33]:
from transformers import BertTokenizer, TFBertModel

yamnet = hub.load("https://tfhub.dev/google/yamnet/1")

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert = TFBertModel.from_pretrained("bert-base-uncased", from_pt=True, use_safetensors=True )

ValueError: Unrecognized keyword arguments passed to TFBertModel: {'use_safetensors': True}

In [34]:
!pip install safetensors

In [29]:
from transformers import BertTokenizer, TFBertModel
import tensorflow_hub as hub

# Load YAMNet audio encoder
yamnet = hub.load("https://tfhub.dev/google/yamnet/1")

# Load BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert = TFBertModel.from_pretrained(
    "bert-base-uncased",
    from_pt=True   # ✅ Converts PyTorch weights to TensorFlow format
)

# Function to get text embedding
def get_question_embedding(question):
    tokens = tokenizer(
        question,
        return_tensors="tf",
        truncation=True,
        padding=True,
        max_length=32
    )
    outputs = bert(**tokens)
    return outputs.pooler_output[0].numpy()   # shape (768,)


AttributeError: 'NoneType' object has no attribute 'items'

In [30]:
print(tf.__version__)

2.20.0


In [ ]:
def build_multimodal_model(audio_dim=1024, text_dim=768):
    #Input Layers
    audio_input = layers.Input(shape=(audio_dim,), name="audio_input")
    text_input = layers.Input(shape=(text_dim,), name="text_input")

    # Audio Reshaping Layers
    x_audio = layers.Dense(256, activation="relu")(audio_input)
    x_audio = layers.Dense(256)(x_audio)

    #Text Reshaping Layer
    x_text = layers.Dense(256, activation="relu")(text_input)

    #Fuse Audio and text
    fusion = layers.Concatenate()([x_audio, x_text])
    fusion = layers.Dense(256, activation="relu")(fusion)
    fusion = layers.Dropout(0.3)(fusion)

    #Output Layer
    output = layers.Dense(1, activation="sigmoid")(fusion)

    model = Model(inputs=[audio_input, text_input], outputs=output)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

model = build_multimodal_model()
model.summary()
model.save("fusion_model.keras")


In [ ]:

from tensorflow.keras.models import load_model
model = load_model("fusion_model_fitted.keras")

In [ ]:
#waveform = load_audio("Recording.wav")
#audio_emb = get_audio_embedding(waveform)  # (1024,)
audio_emb = embeddings
text_emb = get_text_embedding("Is this audio real or fake?")  # (1, 768)

import numpy as np

pred = model.predict({
    "audio_input": audio_emb[None, :],
    "text_input": text_emb.numpy()
})

print("Fake probability:", pred[0][0])



In [ ]:
pred